# NYC Taxi Trip Data — Exploratory Analysis



## 1. Start a Spark session

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = SparkSession.builder.appName("TaxiDataExploration").getOrCreate()
spark

## 2. Load the Parquet file

In [ ]:
RAW_PATH = r"C:\Users\MariamMohsen\Downloads\pyspark\Project\nyc-taxi-fare-intelligence\data\raw\yellow_tripdata_2025-01.parquet"

df = spark.read.parquet(RAW_PATH)
print(f"Rows:    {df.count():,}")
print(f"Columns: {len(df.columns)}")

Rows:    3,475,226
Columns: 20


## 3. Columns and schema



In [4]:
df.printSchema()

root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: long (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- Airport_fee: double (nullable = true)
 |-- cbd_congestion_fee: double (nullable = true)



In [5]:
df.columns

['VendorID',
 'tpep_pickup_datetime',
 'tpep_dropoff_datetime',
 'passenger_count',
 'trip_distance',
 'RatecodeID',
 'store_and_fwd_flag',
 'PULocationID',
 'DOLocationID',
 'payment_type',
 'fare_amount',
 'extra',
 'mta_tax',
 'tip_amount',
 'tolls_amount',
 'improvement_surcharge',
 'total_amount',
 'congestion_surcharge',
 'Airport_fee',
 'cbd_congestion_fee']

## 4. Sample rows

In [6]:
df.show(5, truncate=False, vertical=True)

-RECORD 0------------------------------------
 VendorID              | 1                   
 tpep_pickup_datetime  | 2025-01-01 00:18:38 
 tpep_dropoff_datetime | 2025-01-01 00:26:59 
 passenger_count       | 1                   
 trip_distance         | 1.6                 
 RatecodeID            | 1                   
 store_and_fwd_flag    | N                   
 PULocationID          | 229                 
 DOLocationID          | 237                 
 payment_type          | 1                   
 fare_amount           | 10.0                
 extra                 | 3.5                 
 mta_tax               | 0.5                 
 tip_amount            | 3.0                 
 tolls_amount          | 0.0                 
 improvement_surcharge | 1.0                 
 total_amount          | 18.0                
 congestion_surcharge  | 2.5                 
 Airport_fee           | 0.0                 
 cbd_congestion_fee    | 0.0                 
-RECORD 1-------------------------

## 5. Column types: categorical vs. numeric vs. timestamp



In [7]:
dtypes = df.dtypes
categorical_cols = [c for c, t in dtypes if t == "string"]
numeric_cols = [c for c, t in dtypes if t in ("double", "int", "bigint", "float")]
timestamp_cols = [c for c, t in dtypes if t == "timestamp"]

print("Categorical columns:", categorical_cols)
print("Numeric columns:    ", numeric_cols)
print("Timestamp columns:  ", timestamp_cols)

Categorical columns: ['store_and_fwd_flag']
Numeric columns:     ['VendorID', 'passenger_count', 'trip_distance', 'RatecodeID', 'PULocationID', 'DOLocationID', 'payment_type', 'fare_amount', 'extra', 'mta_tax', 'tip_amount', 'tolls_amount', 'improvement_surcharge', 'total_amount', 'congestion_surcharge', 'Airport_fee', 'cbd_congestion_fee']
Timestamp columns:   []


## 6. Missing values per column



In [8]:
null_counts = df.select(
    [F.count(F.when(F.col(c).isNull(), c)).alias(c) for c in df.columns]
)
null_counts.show(vertical=True, truncate=False)

-RECORD 0-----------------------
 VendorID              | 0      
 tpep_pickup_datetime  | 0      
 tpep_dropoff_datetime | 0      
 passenger_count       | 540149 
 trip_distance         | 0      
 RatecodeID            | 540149 
 store_and_fwd_flag    | 540149 
 PULocationID          | 0      
 DOLocationID          | 0      
 payment_type          | 0      
 fare_amount           | 0      
 extra                 | 0      
 mta_tax               | 0      
 tip_amount            | 0      
 tolls_amount          | 0      
 improvement_surcharge | 0      
 total_amount          | 0      
 congestion_surcharge  | 540149 
 Airport_fee           | 540149 
 cbd_congestion_fee    | 0      



## 7. Summary statistics for numeric columns

In [9]:
df.select(numeric_cols).describe().show(vertical=True, truncate=False)

-RECORD 0------------------------------------
 summary               | count               
 VendorID              | 3475226             
 passenger_count       | 2935077             
 trip_distance         | 3475226             
 RatecodeID            | 2935077             
 PULocationID          | 3475226             
 DOLocationID          | 3475226             
 payment_type          | 3475226             
 fare_amount           | 3475226             
 extra                 | 3475226             
 mta_tax               | 3475226             
 tip_amount            | 3475226             
 tolls_amount          | 3475226             
 improvement_surcharge | 3475226             
 total_amount          | 3475226             
 congestion_surcharge  | 2935077             
 Airport_fee           | 2935077             
 cbd_congestion_fee    | 3475226             
-RECORD 1------------------------------------
 summary               | mean                
 VendorID              | 1.7854283

## 8. Identifying the target column



In [10]:
TARGET_COL = "fare_amount"

df.select(TARGET_COL).describe().show()
df.select(TARGET_COL).summary("min", "25%", "50%", "75%", "max").show()

+-------+------------------+
|summary|       fare_amount|
+-------+------------------+
|  count|           3475226|
|   mean| 17.08180276045484|
| stddev|463.47291781729996|
|    min|            -900.0|
|    max|         863372.12|
+-------+------------------+

+-------+-----------+
|summary|fare_amount|
+-------+-----------+
|    min|     -900.0|
|    25%|        8.6|
|    50%|      12.12|
|    75%|       19.5|
|    max|  863372.12|
+-------+-----------+



## 9. Target sanity check

In [11]:
bad_fare_count = df.where(
    (F.col(TARGET_COL) <= 0) | (F.col(TARGET_COL) > 250)
).count()

print(f"Rows with implausible fare_amount: {bad_fare_count:,} ({bad_fare_count / df.count():.2%} of total)")

Rows with implausible fare_amount: 146,075 (4.20% of total)
